# Open Legal Data Germany — Court Decision Scraper

Collects German court decisions from [de.openlegaldata.io](https://de.openlegaldata.io/) matching keywords
related to parental alienation and child welfare. Used in the comparative legal analysis chapter of the
master's thesis alongside Austrian RIS decisions, ECHR case law, and Reddit discourse.

**API base:** `https://de.openlegaldata.io/api/`  
**No API key required** for read access.  
**Dataset size:** ~420,000 decisions (as of 2024).

---

### German vs Austrian Legal Terminology

Key differences relevant to this thesis:

| Concept | German (DE) | Austrian (AT) |
|---------|-------------|---------------|
| Visitation/contact rights | **Umgangsrecht** | **Kontaktrecht** |
| Custody | **Sorgerecht** | **Obsorge** |
| Child welfare | **Kindeswohl** | **Kindeswohl** (shared) |
| Endangerment of child welfare | **Kindeswohlgefährdung** | **Kindeswohlgefährdung** (shared) |
| Parental alienation | **Elterliche Entfremdung** / **PAS** | **Elterliche Entfremdung** / **PAS** |

---

### Known Limitation: Publication Rate

German courts publish only an estimated **~1.4% of all decisions** (Osterloh-Konrad 2018; Röhl 2014).
Selection bias is substantial: higher courts (BGH, OLG) are overrepresented; family court (Familiengericht)
decisions at Amtsgericht level are largely absent. This platform reflects that bias — it skews toward
legally significant precedent cases rather than routine decisions.

**Methodological implication:** Comparisons with Austrian RIS must account for the much higher publication
rate in Austria (where Rechtssätze and Entscheidungstexte are systematically archived).
Document this caveat explicitly in the thesis methodology section.

---

### API Behaviour (discovered via live testing)

| Detail | Value |
|--------|-------|
| Search endpoint | `GET /api/cases/search/?text={keyword}&page={n}` |
| Search param name | **`text`** (not `q` — using `q` returns HTTP 400) |
| Page size | 10 results per page |
| Date filtering | **Not supported by API** — must filter in Python |
| Full text | Requires `/cases/{id}/` — search results return only `slug` + `snippets` |
| ID lookup | `/cases/?slug={slug}` → get `id` → `/cases/{id}/` → get `content` |
| Content format | HTML — strip with BeautifulSoup `html.parser` |

In [19]:
from __future__ import annotations

import json
import logging
import os
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## Configuration

Edit `START_DATE` / `END_DATE` to extend the collection window.  
For PoC: 2024–2025. For full thesis: `START_DATE = "2000-01-01"`.

In [20]:
START_DATE  = "2018-01-01"   # PoC range — extend to "2000-01-01" for full run
END_DATE    = "2025-12-31"

REQUEST_DELAY    = 1.5   # seconds between API requests
MAX_PAGES        = 50    # max pages to fetch per keyword (10 results/page = 500 max)
CHECKPOINT_EVERY = 25    # save checkpoint after this many new cases with full text

API_BASE     = "https://de.openlegaldata.io/api"
OUTPUT_DIR   = Path("../data/open_legal_data_germany")
OUTPUT_CASES = OUTPUT_DIR / "cases.json"
OUTPUT_KWIC  = OUTPUT_DIR / "kwic.json"
OUTPUT_META  = OUTPUT_DIR / "cases_meta.csv"
CHECKPOINT   = OUTPUT_DIR / "checkpoint.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Output directory: /Users/maksimsmirnov/Desktop/thesis/data/open_legal_data_germany


## Keywords

Each keyword triggers a separate API search. Cases matching multiple keywords are deduplicated
and carry a `matched_keywords` list. **No court or legal-area filter** — which courts and topics
appear is itself an analytical finding.

In [21]:
KEYWORDS = [
    "Entfremdung",
    "Kindeswohl",
    "elterliche Entfremdung",
    "Kontaktrecht",
    "Umgangsrecht",
    "Sorgerechtsentzug",
    "Kindeswohlgefährdung",
    "PAS",
]

## Logging

In [22]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

session = requests.Session()
session.headers.update({
    "User-Agent": "MasterThesis-Research/1.0 (Academic; contact: thesis@example.com)",
    "Accept": "application/json",
})

## API Exploration

Run these cells once to verify the API is reachable and to inspect the live response structure.
The results informed the documented API behaviour table in the title cell.

In [23]:
# ── API root: list all available endpoints ───────────────────────────────────
r = session.get(f"{API_BASE}/", timeout=15)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 200
{
  "laws/search": "https://de.openlegaldata.io/api/laws/search/",
  "cases/search": "https://de.openlegaldata.io/api/cases/search/",
  "cases/stats": "https://de.openlegaldata.io/api/cases/stats/",
  "users": "https://de.openlegaldata.io/api/users/",
  "laws": "https://de.openlegaldata.io/api/laws/",
  "law_books": "https://de.openlegaldata.io/api/law_books/",
  "cases": "https://de.openlegaldata.io/api/cases/",
  "courts": "https://de.openlegaldata.io/api/courts/",
  "cities": "https://de.openlegaldata.io/api/cities/",
  "states": "https://de.openlegaldata.io/api/states/",
  "countries": "https://de.openlegaldata.io/api/countries/",
  "annotation_labels": "https://de.openlegaldata.io/api/annotation_labels/",
  "case_annotations": "https://de.openlegaldata.io/api/case_annotations/",
  "case_markers": "https://de.openlegaldata.io/api/case_markers/"
}


In [24]:
# ── One sample search to confirm correct parameter name and page structure ───
r = session.get(f"{API_BASE}/cases/search/",
                params={"text": "Kindeswohl", "format": "json"}, timeout=15)
data = r.json()
print(f"count={data['count']}, results on page={len(data['results'])}")
print(f"next URL: {data['next']}")
print("\nSearch result fields:")
for k, v in data["results"][0].items():
    print(f"  {k!r}: ({type(v).__name__}) {str(v)[:150]!r}")

count=2841, results on page=10
next URL: https://de.openlegaldata.io/api/cases/search/?format=json&page=2&text=Kindeswohl

Search result fields:
  'slug': (str) 'olgrost-2011-02-11-10-wf-3911'
  'date': (str) '2011-02-11'
  'court': (str) 'OLGROST'
  'court_jurisdiction': (NoneType) 'None'
  'court_level_of_appeal': (str) 'Oberlandesgericht'
  'decision_type': (str) 'Beschluss'
  'snippets': (list) "[{'text': 'Zur Begründung führt er aus, die gemeinsame Sorge entspreche dem <em>Kindeswohl</em> am besten. Die Antragsgegnerin sei mit der alleinigen "


In [25]:
# ── Full case detail: slug → id lookup → content ─────────────────────────────
sample_slug = data["results"][0]["slug"]
print(f"Slug from search: {sample_slug}")

# Step 1: resolve slug to numeric ID
r_list = session.get(f"{API_BASE}/cases/",
                     params={"slug": sample_slug, "format": "json"}, timeout=15)
list_results = r_list.json().get("results", [])
print(f"\nList endpoint fields (no 'content'):")
for k, v in list_results[0].items():
    print(f"  {k!r}: {str(v)[:100]!r}")

case_id = list_results[0]["id"]
print(f"\nCase ID: {case_id}")

# Step 2: fetch full detail by ID
r_detail = session.get(f"{API_BASE}/cases/{case_id}/",
                       params={"format": "json"}, timeout=15)
detail = r_detail.json()
content_raw = detail.get("content", "")
soup = BeautifulSoup(content_raw, "html.parser")
plain = soup.get_text(separator="\n", strip=True)
print(f"\nContent: {len(content_raw)} chars HTML → {len(plain)} chars plain text")
print(f"\nPlain text preview:\n{plain[:500]}")

Slug from search: olgrost-2011-02-11-10-wf-3911

List endpoint fields (no 'content'):
  'id': '94591'
  'slug': 'olgrost-2011-02-11-10-wf-3911'
  'court': "{'id': 483, 'name': 'Oberlandesgericht Rostock', 'slug': 'olgrost', 'city': None, 'state': 10, 'juri"
  'file_number': '10 WF 39/11'
  'date': '2011-02-11'
  'created_date': '2018-11-15T13:30:09Z'
  'updated_date': '2019-02-12T12:46:32Z'
  'type': 'Beschluss'
  'ecli': ''

Case ID: 94591

Content: 9195 chars HTML → 6440 chars plain text

Plain text preview:
Tenor
Die sofortige Beschwerde des Antragstellers gegen den Verfahrenskostenhilfe versagenden Beschluss des Amtsgerichts - Familiengericht - Rostock vom 17.01.2011 wird zurückgewiesen.
Gründe
I.
1
Der Antragsteller ist der Vater des nichtehelich geborenen Kindes, die Antragsgegnerin ist die Kindesmutter. Die Beziehung der beiden ist beendet. Der Antragsteller lebt bei seiner Mutter und seiner Großmutter, die Antragsgegnerin mit dem Kind in einer eigenen Wohnung. Die Antragsgegner

## Helper Functions

In [26]:
def html_to_text(html: str) -> str:
    """Strip HTML tags and return normalised plain text. Uses html.parser (no lxml needed)."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n", strip=True)


def extract_kwic(text: str, keyword: str, window: int = 200) -> list:
    """Extract all occurrences of keyword with ±window character context."""
    contexts = []
    if not text:
        return contexts
    text_lower = text.lower()
    kw_lower = keyword.lower()
    start = 0
    while True:
        idx = text_lower.find(kw_lower, start)
        if idx == -1:
            break
        ctx_start = max(0, idx - window)
        ctx_end = min(len(text), idx + len(keyword) + window)
        contexts.append({
            "keyword":  keyword,
            "position": idx,
            "context":  text[ctx_start:ctx_end].strip(),
        })
        start = idx + 1
    return contexts


def parse_year(date_str: str):
    """Extract year from ISO date string like '2024-03-15'."""
    if not date_str:
        return None
    try:
        return int(str(date_str)[:4])
    except (ValueError, TypeError):
        return None


def in_date_range(date_str: str) -> bool:
    """Return True if date_str falls within [START_DATE, END_DATE]."""
    if not date_str:
        return False
    return START_DATE <= date_str[:10] <= END_DATE

In [27]:
def api_get(url: str, params: dict = None, retries: int = 3):
    """
    GET request with retry + exponential backoff.
    Returns parsed JSON dict or None on persistent failure.
    """
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            r = session.get(url, params=params, timeout=20)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.HTTPError as e:
            log.warning(f"HTTP {r.status_code} (attempt {attempt+1}): {url}")
            if r.status_code == 429:
                time.sleep(30 * (attempt + 1))
            elif r.status_code >= 500:
                time.sleep(5 * (attempt + 1))
            else:
                return None   # 4xx client error — no point retrying
        except (requests.exceptions.RequestException, ValueError) as e:
            log.warning(f"Request error (attempt {attempt+1}): {e}")
            time.sleep(3 * (attempt + 1))
    log.error(f"All {retries} attempts failed: {url}")
    return None

In [28]:
def search_keyword(keyword: str) -> list:
    """
    Paginate through /cases/search/?text={keyword} and collect all results.
    Date filtering is done in Python (API does not support date filters on search).
    Returns a list of search-result dicts, each with: slug, date, court,
    court_level_of_appeal, decision_type, snippets.
    """
    results = []
    url = f"{API_BASE}/cases/search/"
    params = {"text": keyword, "format": "json"}
    page = 0

    while url and page < MAX_PAGES:
        data = api_get(url, params)
        params = None   # subsequent calls use the full `next` URL
        page += 1

        if not data:
            log.warning(f"[{keyword}] No data on page {page}")
            break

        page_results = data.get("results", [])
        if isinstance(page_results, dict):   # single result edge case
            page_results = [page_results]

        added = 0
        for r in page_results:
            if in_date_range(r.get("date", "")):
                results.append(r)
                added += 1

        total = data.get("count", "?")
        log.info(f"[{keyword}] page {page}/{(total // 10 + 1) if isinstance(total, int) else '?'} "
                 f"— {added} in date range (page had {len(page_results)}, total hits: {total})")

        # Stop early if ALL results on this page pre-date START_DATE
        # (API orders by relevance/date descending, so once we see old dates we can stop)
        dates_on_page = [r.get("date", "") for r in page_results if r.get("date")]
        if dates_on_page and max(dates_on_page) < START_DATE:
            log.info(f"[{keyword}] All results on page {page} pre-date {START_DATE}, stopping.")
            break

        url = data.get("next")  # None when last page reached

    log.info(f"[{keyword}] → {len(results)} results in date range")
    return results

In [29]:
def fetch_full_case(slug: str):
    """
    Fetch complete case data for a given slug.

    Two-step process:
      1. GET /cases/?slug={slug}  → resolves slug to numeric ID + gets metadata
      2. GET /cases/{id}/         → gets full content (HTML)

    Returns a flat dict with all fields, or None on failure.
    `content` field is stripped to plain text.
    """
    # Step 1: slug → id
    list_data = api_get(f"{API_BASE}/cases/", params={"slug": slug, "format": "json"})
    if not list_data:
        log.warning(f"  Could not look up slug: {slug}")
        return None

    list_results = list_data.get("results", [])
    if not list_results:
        log.warning(f"  Slug not found in list endpoint: {slug}")
        return None

    meta = list_results[0]
    case_id = meta.get("id")
    if not case_id:
        return None

    # Step 2: fetch full detail (includes 'content')
    detail = api_get(f"{API_BASE}/cases/{case_id}/", params={"format": "json"})
    if not detail:
        log.warning(f"  Could not fetch detail for id={case_id} ({slug})")
        # Return metadata without text
        detail = {}

    # Merge list metadata + detail, normalise content to plain text
    court = meta.get("court") or detail.get("court") or {}
    if isinstance(court, str):
        court = {"slug": court, "name": court}

    content_html = detail.get("content", "")
    content_text = html_to_text(content_html)

    return {
        "id":               case_id,
        "slug":             slug,
        "court_name":       court.get("name", ""),
        "court_slug":       court.get("slug", ""),
        "court_level":      court.get("level_of_appeal", ""),
        "court_jurisdiction": court.get("jurisdiction", ""),
        "file_number":      meta.get("file_number", ""),
        "date":             meta.get("date", ""),
        "year":             parse_year(meta.get("date", "")),
        "decision_type":    meta.get("type", detail.get("type", "")),
        "ecli":             meta.get("ecli", detail.get("ecli", "")),
        "content":          content_text,
        "text_length":      len(content_text),
        "matched_keywords": [],   # populated by caller
    }

In [30]:
def load_checkpoint() -> tuple:
    """Return (cases_by_slug dict, kwic_list) from checkpoint, or empty defaults."""
    if not CHECKPOINT.exists():
        return {}, []
    try:
        with open(CHECKPOINT, encoding="utf-8") as f:
            data = json.load(f)
        cases = {c["slug"]: c for c in data.get("cases", []) if c.get("slug")}
        kwic  = data.get("kwic", [])
        log.info(f"Checkpoint loaded: {len(cases)} cases, {len(kwic)} KWIC entries")
        return cases, kwic
    except Exception as e:
        log.warning(f"Could not load checkpoint ({e}) — starting fresh")
        return {}, []


def save_checkpoint(cases_by_slug: dict, kwic_list: list) -> None:
    """Write current state to checkpoint file."""
    try:
        data = {
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "cases":    list(cases_by_slug.values()),
            "kwic":     kwic_list,
        }
        with open(CHECKPOINT, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        log.info(f"Checkpoint saved: {len(cases_by_slug)} cases, {len(kwic_list)} KWIC entries")
    except Exception as e:
        log.warning(f"Could not save checkpoint: {e}")

## Phase 1 — Collect Search Results

For each keyword, search the API and collect matching case slugs within the date range.
Cases matching multiple keywords are deduplicated; `matched_keywords` tracks all matching terms.

In [31]:
# Load checkpoint — resume from interrupted run if available
cases_by_slug, kwic_list = load_checkpoint()
log.info(f"Starting with {len(cases_by_slug)} cases already in checkpoint")

# slug → set of matched keywords (accumulated across keyword loops)
slug_to_keywords: dict = {
    slug: set(c.get("matched_keywords", []))
    for slug, c in cases_by_slug.items()
}
# Also accumulate slugs found in this run's search (may not yet have full text)
search_hits: dict = {}   # slug → partial search result dict
keyword_raw_counts: dict = {}

14:45:15  INFO      Checkpoint loaded: 0 cases, 0 KWIC entries
14:45:15  INFO      Starting with 0 cases already in checkpoint


In [32]:
for i, keyword in enumerate(KEYWORDS):
    results = search_keyword(keyword)
    keyword_raw_counts[keyword] = len(results)

    for r in results:
        slug = r.get("slug")
        if not slug:
            continue
        if slug not in search_hits:
            search_hits[slug] = r
        slug_to_keywords.setdefault(slug, set()).add(keyword)

    if i < len(KEYWORDS) - 1:
        log.info(f"Sleeping {REQUEST_DELAY}s before next keyword …")
        time.sleep(REQUEST_DELAY)

log.info(f"Total unique slugs found across all keywords: {len(search_hits)}")
print(f"\nRaw hits per keyword:")
for kw, n in keyword_raw_counts.items():
    print(f"  {n:>5}  {kw}")

14:45:16  INFO      [Entfremdung] page 1/42 — 1 in date range (page had 10, total hits: 419)
14:45:18  INFO      [Entfremdung] page 2/42 — 3 in date range (page had 10, total hits: 419)
14:45:20  INFO      [Entfremdung] page 3/42 — 1 in date range (page had 10, total hits: 419)
14:45:21  INFO      [Entfremdung] page 4/42 — 1 in date range (page had 10, total hits: 419)
14:45:23  INFO      [Entfremdung] page 5/42 — 1 in date range (page had 10, total hits: 419)
14:45:24  INFO      [Entfremdung] page 6/42 — 2 in date range (page had 10, total hits: 419)
14:45:26  INFO      [Entfremdung] page 7/42 — 3 in date range (page had 10, total hits: 419)
14:45:27  INFO      [Entfremdung] page 8/42 — 1 in date range (page had 10, total hits: 419)
14:45:29  INFO      [Entfremdung] page 9/42 — 3 in date range (page had 10, total hits: 419)
14:45:30  INFO      [Entfremdung] page 10/42 — 0 in date range (page had 10, total hits: 419)
14:45:30  INFO      [Entfremdung] All results on page 10 pre-date 201


Raw hits per keyword:
     16  Entfremdung
      0  Kindeswohl
      1  elterliche Entfremdung
      0  Kontaktrecht
      2  Umgangsrecht
     35  Sorgerechtsentzug
     40  Kindeswohlgefährdung
     27  PAS


## Phase 2 — Fetch Full Texts

For each unique slug found in Phase 1 that isn't already in the checkpoint,
fetch the full case (2 API requests: slug → id, then id → content).
Checkpoints every 25 new cases.

In [33]:
new_since_checkpoint = 0
slugs_to_fetch = [s for s in search_hits if s not in cases_by_slug]
log.info(f"Slugs to fetch: {len(slugs_to_fetch)} (skipping {len(cases_by_slug)} already in checkpoint)")

for i, slug in enumerate(slugs_to_fetch):
    log.info(f"[{i+1}/{len(slugs_to_fetch)}] Fetching: {slug}")
    case = fetch_full_case(slug)

    if case is None:
        # Store a stub so we don't retry on the next run
        case = {
            "id": None, "slug": slug, "content": "", "text_length": 0,
            "matched_keywords": [], "date": search_hits[slug].get("date", ""),
            "year": parse_year(search_hits[slug].get("date", "")),
            "court_name": search_hits[slug].get("court", ""),
            "court_level": search_hits[slug].get("court_level_of_appeal", ""),
            "decision_type": search_hits[slug].get("decision_type", ""),
            "file_number": "", "ecli": "",
            "court_slug": "", "court_jurisdiction": "",
        }

    case["matched_keywords"] = sorted(slug_to_keywords.get(slug, set()))
    cases_by_slug[slug] = case
    new_since_checkpoint += 1

    if new_since_checkpoint >= CHECKPOINT_EVERY:
        save_checkpoint(cases_by_slug, kwic_list)
        new_since_checkpoint = 0

# Final checkpoint after loop
save_checkpoint(cases_by_slug, kwic_list)

# Update matched_keywords for cases already in checkpoint (new keyword runs may have added matches)
for slug, kw_set in slug_to_keywords.items():
    if slug in cases_by_slug:
        existing = set(cases_by_slug[slug].get("matched_keywords", []))
        cases_by_slug[slug]["matched_keywords"] = sorted(existing | kw_set)

n_with_text = sum(1 for c in cases_by_slug.values() if c.get("text_length", 0) > 100)
print(f"\nCases collected: {len(cases_by_slug)}  |  with full text: {n_with_text}")

14:46:42  INFO      Slugs to fetch: 106 (skipping 0 already in checkpoint)
14:46:42  INFO      [1/106] Fetching: olgk-2018-11-13-10-wf-16418
14:46:45  INFO      [2/106] Fetching: bverfg-2019-08-12-1-bvr-174218
14:46:48  INFO      [3/106] Fetching: bgh-2021-05-06-iii-zr-7220
14:46:51  INFO      [4/106] Fetching: olghh-2021-02-02-12-wf-521
14:46:54  INFO      [5/106] Fetching: verfgsn-2020-12-03-vf-205-iv-20-hsv
14:46:57  INFO      [6/106] Fetching: vghbw-2020-03-02-11-s-229318
14:47:00  INFO      [7/106] Fetching: kg--2018-12-14-5-ws-20218-vollz
14:47:03  INFO      [8/106] Fetching: olgsh-2020-05-25-10-wf-7720
14:47:06  INFO      [9/106] Fetching: vg-dusseldorf-2018-02-01-18-l-9618
14:47:10  INFO      [10/106] Fetching: verfgbe-2022-12-14-8421
14:47:13  INFO      [11/106] Fetching: ovgbebb-2020-07-08-ovg-11-n-7218
14:47:16  INFO      [12/106] Fetching: bsg-2019-03-12-b-13-r-2717-r
14:47:19  INFO      [13/106] Fetching: olgdres-2020-07-15-3-uf-1420
14:47:22  INFO      [14/106] Fetching: 


Cases collected: 106  |  with full text: 105


## Phase 3 — KWIC Extraction

For each case with full text, extract every occurrence of every matched keyword
with a ±200-character context window.

In [34]:
kwic_done_slugs = {e["slug"] for e in kwic_list}
new_kwic = 0

for slug, case in cases_by_slug.items():
    if slug in kwic_done_slugs:
        continue

    text = case.get("content", "") or ""
    for keyword in case.get("matched_keywords", []):
        for hit in extract_kwic(text, keyword):
            kwic_list.append({
                "slug":        slug,
                "court_name":  case.get("court_name", ""),
                "court_level": case.get("court_level", ""),
                "year":        case.get("year"),
                "date":        case.get("date", ""),
                "keyword":     hit["keyword"],
                "position":    hit["position"],
                "context":     hit["context"],
            })
            new_kwic += 1

    kwic_done_slugs.add(slug)

save_checkpoint(cases_by_slug, kwic_list)
print(f"KWIC extraction complete: {len(kwic_list)} total contexts ({new_kwic} new this run)")

14:56:46  INFO      Checkpoint saved: 106 cases, 700 KWIC entries


KWIC extraction complete: 700 total contexts (700 new this run)


## Phase 4 — Save Outputs

In [35]:
all_cases = list(cases_by_slug.values())

# cases.json — full dataset including text
with open(OUTPUT_CASES, "w", encoding="utf-8") as f:
    json.dump(all_cases, f, ensure_ascii=False, indent=2)
print(f"Cases saved  → {OUTPUT_CASES}  ({len(all_cases)} cases)")

# kwic.json — flat list of keyword-in-context entries
with open(OUTPUT_KWIC, "w", encoding="utf-8") as f:
    json.dump(kwic_list, f, ensure_ascii=False, indent=2)
print(f"KWIC saved   → {OUTPUT_KWIC}  ({len(kwic_list)} entries)")

# cases_meta.csv — metadata without full text (easy to open in Excel)
meta_cols = ["id", "slug", "court_name", "court_level", "court_jurisdiction",
             "file_number", "date", "year", "decision_type", "ecli",
             "text_length", "matched_keywords"]
rows = []
for c in all_cases:
    row = {col: c.get(col, "") for col in meta_cols}
    row["matched_keywords"] = "; ".join(c.get("matched_keywords") or [])
    rows.append(row)
pd.DataFrame(rows).to_csv(OUTPUT_META, index=False, encoding="utf-8-sig")
print(f"Meta CSV     → {OUTPUT_META}")

Cases saved  → ../data/open_legal_data_germany/cases.json  (106 cases)
KWIC saved   → ../data/open_legal_data_germany/kwic.json  (700 entries)
Meta CSV     → ../data/open_legal_data_germany/cases_meta.csv


## Summary Statistics

In [36]:
n_total     = len(all_cases)
n_with_text = sum(1 for c in all_cases if (c.get("text_length") or 0) > 100)

year_counter   = Counter(c["year"] for c in all_cases if c.get("year"))
level_counter  = Counter(c.get("court_level") or "unknown" for c in all_cases)
kwic_per_kw    = Counter(e["keyword"] for e in kwic_list)

# Cases per keyword (by matched_keywords field)
kw_case_counts = {
    kw: sum(1 for c in all_cases if kw in (c.get("matched_keywords") or []))
    for kw in KEYWORDS
}

# Keyword-in-fulltext vs keyword-in-search (search finds via index, text may differ)
kw_in_text = {
    kw: sum(
        1 for c in all_cases
        if kw in (c.get("matched_keywords") or []) and
        kw.lower() in (c.get("content") or "").lower()
    )
    for kw in KEYWORDS
}

print(f"{'='*60}")
print(f"Total unique cases        : {n_total}")
print(f"Cases with full text      : {n_with_text}")
print(f"Cases without full text   : {n_total - n_with_text}")
print(f"Total KWIC contexts       : {len(kwic_list)}")
print(f"{'-'*60}")
print("Cases per keyword (matched_keywords list):")
for kw in KEYWORDS:
    n_matched = kw_case_counts.get(kw, 0)
    n_text    = kw_in_text.get(kw, 0)
    print(f"  {n_matched:>5}  {kw}  (keyword found in full text: {n_text})")
print(f"{'-'*60}")
print("KWIC contexts per keyword:")
for kw in KEYWORDS:
    print(f"  {kwic_per_kw.get(kw, 0):>5}  {kw}")
print(f"{'-'*60}")
print("Cases per year:")
for yr, cnt in sorted(year_counter.items(), reverse=True):
    print(f"  {yr}: {cnt}")
print(f"{'-'*60}")
print("Cases per court level (Instanz):")
for level, cnt in level_counter.most_common():
    print(f"  {cnt:>5}  {level}")
print(f"{'-'*60}")
print("Text length distribution (chars):")
lengths = [c["text_length"] for c in all_cases if (c.get("text_length") or 0) > 0]
if lengths:
    s = pd.Series(lengths)
    print(s.describe().to_string())
print(f"{'='*60}")

Total unique cases        : 106
Cases with full text      : 105
Cases without full text   : 1
Total KWIC contexts       : 700
------------------------------------------------------------
Cases per keyword (matched_keywords list):
     16  Entfremdung  (keyword found in full text: 16)
      0  Kindeswohl  (keyword found in full text: 0)
      1  elterliche Entfremdung  (keyword found in full text: 0)
      0  Kontaktrecht  (keyword found in full text: 0)
      2  Umgangsrecht  (keyword found in full text: 2)
     35  Sorgerechtsentzug  (keyword found in full text: 34)
     40  Kindeswohlgefährdung  (keyword found in full text: 40)
     27  PAS  (keyword found in full text: 27)
------------------------------------------------------------
KWIC contexts per keyword:
     35  Entfremdung
      0  Kindeswohl
      0  elterliche Entfremdung
      0  Kontaktrecht
     83  Umgangsrecht
    115  Sorgerechtsentzug
    337  Kindeswohlgefährdung
    130  PAS
----------------------------------------

exploration

In [41]:
# Run these quick checks on your German data
import json

with open("../data/open_legal_data_germany/cases.json", "r") as f:
    cases = json.load(f)

# 1. Which keywords actually hit?
from collections import Counter
kw_counts = Counter()
for c in cases:
    for kw in c.get("matched_keywords", []):
        kw_counts[kw] += 1
print("Keyword distribution:")
for kw, count in kw_counts.most_common():
    print(f"  {kw}: {count}")

# 2. Which courts?
courts = Counter(c.get("court", "?") for c in cases)
print("\nCourt distribution:")
for court, count in courts.most_common():
    print(f"  {court}: {count}")

# 3. Do all cases actually contain your keywords in the text?
# (API search might match metadata, not just body text)
for kw in ["Entfremdung", "Kindeswohl"]:
    has_kw = sum(1 for c in cases if kw.lower() in c.get("content", "").lower())
    print(f"\n'{kw}' in full text: {has_kw} / {len(cases)}")

Keyword distribution:
  Kindeswohlgefährdung: 40
  Sorgerechtsentzug: 35
  PAS: 27
  Entfremdung: 16
  Umgangsrecht: 2
  elterliche Entfremdung: 1

Court distribution:
  ?: 106

'Entfremdung' in full text: 19 / 106

'Kindeswohl' in full text: 63 / 106


In [39]:
case = cases[0]
for k, v in case.items():
    if isinstance(v, str) and len(v) > 500:
        print(f"Field '{k}': {len(v)} chars")

Field 'content': 4402 chars


In [43]:
# Family law relevance filter
FAMILY_TERMS = ["kindeswohl", "sorgerecht", "umgangsrecht", 
                "obsorge", "kontaktrecht", "besuchsrecht",
                "elternteil", "kind", "mutter", "vater",
                "familienrecht", "trennung"]

def is_family_law_relevant(text):
    """Check if text is actually about family law, not just 
    coincidentally containing a keyword."""
    if not text:
        return False
    text_lower = text.lower()
    matches = sum(1 for term in FAMILY_TERMS if term in text_lower)
    return matches >= 2  # at least 2 family law terms present

# Filter
relevant = [c for c in cases if is_family_law_relevant(c.get("content", ""))]
print(f"Family law relevant: {len(relevant)} / {len(cases)}")

Family law relevant: 79 / 106


## Troubleshooting

### HTTP 400 on search
The search endpoint requires the parameter to be named **`text`**, not `q`:
```
GET /api/cases/search/?text=Kindeswohl   ✓
GET /api/cases/search/?q=Kindeswohl     ✗  → HTTP 400
```

### Zero results in date range
Open Legal Data publishes decisions with significant delay. Decisions from 2024–2025 may not yet
be indexed. Try extending `START_DATE` to `"2020-01-01"` or earlier for more results.

### Empty `content` field
Not all cases on the platform have full text. Some entries have only metadata (court, date,
file number). These will have `text_length = 0`. Check `cases_meta.csv` for the distribution.

### Very few results for some keywords
`PAS` is a 3-letter acronym — the search engine may treat it as a stop word or return
false positives. `Sorgerechtsentzug` is a compound noun that may appear as sub-words
(`Sorgerecht` + `entzug`) in older texts.

### Rate limiting / slow runs
Open Legal Data is a volunteer-run non-profit. Keep `REQUEST_DELAY ≥ 1.5` and avoid
running more than a few thousand requests in a session. The checkpoint ensures you can
resume without re-fetching already-collected cases.

### Comparison with Austrian RIS
Austrian RIS (via `ris_ogd_api_scraper_v2.ipynb`) uses `Entfremdung` and `Kindeswohl`
as the primary German-language terms. Comparing frequency and framing across RIS and
this dataset is a core thesis finding — but note the publication-rate caveat above.
Austrian courts publish near-comprehensively via RIS; German courts publish selectively.
Frame frequency comparisons as *discourse within published case law*, not as population estimates.